##Création de l'Environnement de Jeu


In [1]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import torch
import random

In [14]:
class ConnectFourEnv:
    # Constructeur de la classe, initialisation de l'environnement avec des dimensions par défaut (6 lignes, 7 colonnes)
    def __init__(self, rows=6, cols=7):
        self.rows = rows  # Définition du nombre de lignes du plateau
        self.cols = cols  # Définition du nombre de colonnes du plateau
        self.board = np.zeros((rows, cols))  # Initialisation du plateau de jeu, rempli de zéros (cases vides)
        self.players = [1, 2]  # Définition des deux joueurs (1 et 2)
        self.current_player = random.choice(self.players)  # Choix aléatoire du joueur actuel
        self.num_moves = 0  # Nombre de coups joués jusqu'à présent
        self.max_moves = rows * cols  # Nombre maximum de coups avant la fin du jeu (plateau rempli)

    # Méthode pour réinitialiser l'environnement (plateau et joueur)
    def reset(self):
        self.board = np.zeros((self.rows, self.cols))  # Initialisation du plateau de jeu, rempli de zéros (cases vides)
        self.num_moves = 0  # Nombre de coups joués jusqu'à présent
        self.current_player = random.choice(self.players)  # Choix aléatoire du joueur actuel
        return self.board

    # Méthode pour obtenir les mouvements valides (colonnes où il reste de la place)
    def get_valid_moves(self):
        return [col for col in range(self.cols) if self.board[self.rows - 1, col] == 0]


    # Méthode qui exécute un coup et met à jour l'état du jeu
    def step(self, action):
        if action not in self.get_valid_moves():
            raise ValueError(f"Invalid action: Column {action} is full or out of range.")
        
        placed_row = None  # Variable pour stocker la ligne où le jeton est placé
        for row in range(self.rows):
            if self.board[row, action] == 0:
                self.board[row, action] = self.current_player
                placed_row = row
                break

        self.num_moves += 1

        # Check if the game has been won
        has_won = self.is_win(placed_row, action)
        is_done = has_won or self.num_moves >= self.max_moves
        reward = 1 if has_won else 0
        winner = self.current_player if has_won else None  # Stocke le joueur gagnant si victoire

        # Switch player
        self.current_player = 3 - self.current_player

        return self.board, reward, is_done, winner


    # Méthode pour vérifier si un joueur a gagné après un coup
    def is_win(self, row, col):
        def check_direction(delta_row, delta_col):
            count = 1
            for d in [-1, 1]:
                r, c = row, col
                while True:
                    r += d * delta_row
                    c += d * delta_col
                    if 0 <= r < self.rows and 0 <= c < self.cols and self.board[r, c] == self.current_player:
                        count += 1
                    else:
                        break
            return count >= 4

        # Check all directions: horizontal, vertical, and two diagonals
        return (
            check_direction(0, 1) or  # Horizontal
            check_direction(1, 0) or  # Vertical
            check_direction(1, 1) or  # Diagonal
            check_direction(1, -1)    # Diagonal
        )

    # Méthode pour afficher l'état actuel du plateau de jeu
    def render(self):
        plt.clf()
        fig, ax = plt.subplots(figsize=(7, 6))
        ax.set_xlim(-0.5, self.cols - 0.5)
        ax.set_ylim(-0.5, self.rows - 0.5)
        ax.set_xticks(range(self.cols))
        ax.set_yticks(range(self.rows))
        ax.set_xticklabels(range(self.cols))
        ax.set_yticklabels(range(self.rows - 1, -1, -1))
        ax.grid(True, which='major', color='black', linestyle='-', linewidth=0.5)

        for row in range(self.rows):
            for col in range(self.cols):
                cell_value = self.board[row, col]
                color = 'white' if cell_value == 0 else 'red' if cell_value == 1 else 'yellow'
                circle = plt.Circle((col, row), 0.4, color=color, ec='black')
                ax.add_patch(circle)

        player1_patch = mpatches.Patch(color='red', label='Player 1')
        player2_patch = mpatches.Patch(color='yellow', label='Player 2')
        ax.legend(handles=[player1_patch, player2_patch], loc='upper right')
        
        plt.show()

# Création d'un Agent Simple

In [3]:
class RandomAgent:
    def __init__(self, player_id):
        self.player_id = player_id

    def select_action(self, board, valid_moves):
        return random.choice(valid_moves)

## Fonction Play Game

In [15]:
# Fonction pour jouer une partie complète entre deux joueurs
def play_game(agent1, agent2):
    env = ConnectFourEnv()
    env.reset()
    done = False
    turn = 0  # Player 1 starts first


    while not done:
        current_agent = agent1 if env.current_player == 1 else agent2
        valid_moves = env.get_valid_moves()

        # Obtenir l'action du joueur
        if isinstance(current_agent, RandomAgent) or isinstance(current_agent, SmarterAgent):  # Agent joue
            action = current_agent.select_action(env.board, valid_moves)
            print(f"Player {env.current_player} (Agent) chooses column {action}")
        else:  # Joueur humain joue
            print(f"Valid moves: {valid_moves}")
            try:
                action = int(input(f"Player {env.current_player}, choose a column: "))
                if action not in valid_moves:
                    raise ValueError("Invalid move. Try again.")
            except ValueError as e:
                print(e)
                continue

        # Appliquer l'action
        _, reward, done, winner = env.step(action)

        # Afficher le plateau
        env.render()

        # Vérifier le résultat
        if reward > 0:
            print(f"Player {winner} wins!")
        elif done:
            print("It's a draw!")
            
        turn += 1

In [ ]:
agent = RandomAgent(player_id=2)  # Agent joue comme joueur 2
play_game(None, agent)  # Joueur humain contre agent

# Création d'un Agent intelligent

In [16]:
# Smarter Agent that selects a winning move, blocks opponent's winning move, or picks a random move
class SmarterAgent:
    def __init__(self, player_id):
        self.player_id = player_id
    
    def select_action(self, board, valid_moves):
        # Check for a winning move for itself
        for action in valid_moves:
            temp_board = board.copy()
            simulated_row = None
            for row in range(temp_board.shape[0]):
                if temp_board[row, action] == 0:
                    temp_board[row, action] = self.player_id
                    simulated_row = row
                    break
            if ConnectFourEnv().is_win(simulated_row, action):
                return action

        # Check for a blocking move against opponent
        for action in valid_moves:
            temp_board = board.copy()
            simulated_row = None
            for row in range(temp_board.shape[0]):
                if temp_board[row, action] == 0:
                    temp_board[row, action] = 3 - self.player_id
                    simulated_row = row
                    break
            if ConnectFourEnv().is_win(simulated_row, action):
                return action

        # Otherwise, pick a random valid move
        return random.choice(valid_moves)

 # Heurstic 1 : One-Step Lookahead


In [ ]:
def count_windows(grid, size, mark, config):
    """Count the number of windows of a given size for a specific player."""

def get_heuristic_q1(grid, col, mark, config):
    """Compute a heuristic score based on different window sizes."""

class HeuristicAgent:
    def __init__(self, player_id):
        self.player_id = player_id

    def get_action(self, env):
        # completer


        return best_move

 # Heurstic 2 : A closer look


# Amélioration de l'Agent avec Deep Reinforcement Learning